# 02 · Explore the MBTA API for one bus route
Schedules, Predictions and Routes for a route that appears in the files (route `01` in the
files is `1` in the API). Small requests, status code first.

In [ ]:
import os
from datetime import date

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv("../.env")
BASE = "https://api-v3.mbta.com"
HEADERS = {"x-api-key": os.environ["MBTA_API_KEY"]} if os.environ.get("MBTA_API_KEY") else {}
TODAY = date.today().isoformat()

## Routes: the id, short name and direction names

In [ ]:
r = requests.get(f"{BASE}/routes/1", headers=HEADERS)
print(r.status_code)
attrs = r.json()["data"]["attributes"]
attrs["short_name"], attrs["direction_names"]

**Notes:** which API direction_id is Inbound?

In [ ]:
r = requests.get(f"{BASE}/routes", params={"filter[type]": "3"}, headers=HEADERS)
routes = pd.DataFrame([{"id": d["id"], "short_name": d["attributes"]["short_name"]} for d in r.json()["data"]])
routes[routes["short_name"].str.startswith(("SL", "CT"))]

**Notes:** how do SL / CT routes map? Which file routes have no API route?

## Schedules: one day of planned service

In [ ]:
r = requests.get(f"{BASE}/schedules", params={"filter[route]": "1", "filter[date]": TODAY}, headers=HEADERS)
print(r.status_code, "records:", len(r.json()["data"]))
r.json()["data"][0]

In [ ]:
data = r.json()["data"]
sched = pd.DataFrame([{
    "direction_id": d["attributes"]["direction_id"],
    "stop_id": d["relationships"]["stop"]["data"]["id"],
    "trip_id": d["relationships"]["trip"]["data"]["id"],
    "arrival_time": d["attributes"]["arrival_time"],
    "departure_time": d["attributes"]["departure_time"],
    "timepoint": d["attributes"].get("timepoint"),
} for d in data])
print("timepoint rows:", sched["timepoint"].sum(), "of", len(sched))
sched.head()

**Notes:** which API field gives each file column? Which file column has no API answer?

## Predictions: the live estimate (not an actual)

In [ ]:
r = requests.get(f"{BASE}/predictions", params={"filter[route]": "1", "page[limit]": "3"}, headers=HEADERS)
print(r.status_code)
[d["attributes"] for d in r.json()["data"]]

**Notes:** why can't a prediction be used as `actual`?